# Project - Airline AI Assistant

Bringing together what we've learned to make an AI Customer Support assistant for an Airline.

In [1]:
# Imports:

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import json

In [2]:
# Initializing API Calling to LLM:

load_dotenv(override= True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print('Open AI API key found!!')
else:
    print('Open AI API key not found!!')

MODEL = 'gpt-4.1-mini'
openai = OpenAI()

Open AI API key found!!


In [3]:
# Defining System Prompt:

system_message = '''
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
'''

In [4]:
# Defining Callback Function:

def chat(message, history):

    history = [{'role': h['role'], 'content': h['content']} for h in history]

    messages = [{'role': 'system', 'content': system_message}] + history + [{'role': 'user', 'content': message}]

    response = openai.chat.completions.create(model= MODEL, messages= messages)

    return response.choices[0].message.content

In [5]:
gr.ChatInterface(fn= chat).launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky. We're giving it the power to run code on our machine?

Well, kinda.

In [6]:
# Making a Simple and useful function:

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f'Tool called for city {destination_city}')
    price = ticket_prices.get(destination_city.lower(), 'Unknown Ticket Price')

    return f'Price of a ticket to {destination_city} is {price}'

In [7]:
get_ticket_price('Mehsana')

Tool called for city Mehsana


'Price of a ticket to Mehsana is Unknown Ticket Price'

In [8]:
get_ticket_price('LonDon')

Tool called for city LonDon


'Price of a ticket to LonDon is $799'

In [9]:
# Writing a Dictionary to Describe above Function to LLM:

# There's a particular dictionary structure that's required to describe our function:

price_function = {
    'name': 'get_ticket_price',
    'description': 'Get the Price of a return ticket to the destination city.',
    'parameters': {
        'type': 'object',
        'properties': {
            'destination_city': {
                'type': 'string',
                'description': 'The city that the customer wants to travel to.',
            },
        },
        'required': ['destination_city'],
        'additionalProperties': False,
    }
}

In [10]:
# And this is included in a list of tools:

tools = [{'type': 'function', 'function': price_function}]

# Understanding `finish_reason`

When building applications or Agentic workflows with LLMs, the `finish_reason` acts as the steering wheel for your entire program. It is a tiny string attached to the response that tells your code exactly **why the LLM stopped generating text**.

### The Object Path
To extract this string, you look at `response.choices[0].finish_reason`:
* **`response`**: This is the massive package of data the API sends back to you. It contains the text, but also metadata like how many tokens you used, the model ID, and timestamps.
* **`.choices[0]`**: The API can theoretically generate multiple different responses at once. This grabs the very first (and usually only) generated response from the list.
* **`.finish_reason`**: The specific condition that caused the model to stop writing.

---

### Common `finish_reason` Values

#### 1. `"stop"` (The Good Ending)
This means the LLM finished its thought naturally. It generated a complete response, hit a natural stopping point (like a period at the end of a concluding sentence), and stopped.
* **What your code should do:** Print the response to the user.

#### 2. `"length"` (The Cut-Off)
This means the model was still talking, but it hit the `max_tokens` limit that you set in your API call, or it hit the absolute maximum context window of the model itself. The output will be cut off mid-sentence.
* **What your code should do:** Either increase the `max_tokens` limit or warn the user that the response was truncated.

#### 3. `"tool_calls"` (The Agent Trigger)
If the LLM decides it needs to use a tool (like checking the weather or running a Python script), it stops generating standard text and sets the finish reason to `"tool_calls"`.
* **What your code should do:** This is the "Pause and Request" step. When your code sees this, it knows it shouldn't show the text to the user. Instead, it needs to look at the tool request, run your local function, and send the result back to the LLM.

#### 4. `"content_filter"` (The Guardrail)
This means the model started generating text, but internal safety filters flagged the content (e.g., it was generating something toxic or dangerous) and abruptly pulled the plug.
* **What your code should do:** Return a polite error message to the user saying the request couldn't be completed due to safety guidelines.

---

### Practical Implementation

In a production environment, you use an `if/elif` block to check the `finish_reason` before taking any action. Here is the standard logic flow:

```python
# Assume 'response' is the object returned from the OpenAI API call
finish_reason = response.choices[0].finish_reason
message = response.choices[0].message

if finish_reason == "stop":
    # Safe to show the user
    print("Agent:", message.content)

elif finish_reason == "tool_calls":
    # Do NOT show the user. Trigger the tool logic.
    print("Agent is requesting a tool...")
    # execute_tool(message.tool_calls)

elif finish_reason == "length":
    # Warn the user about the cut-off
    print("Warning: Response cut off due to token limits.")
    print("Agent:", message.content)

elif finish_reason == "content_filter":
    # Handle the safety block
    print("Error: Response blocked by safety filters.")
```

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [22]:
def chat(message, history):

    history = [{'role': h['role'], 'content': h['content']} for h in history]

    messages = [{'role': 'system', 'content': system_message}] + history + [{'role': 'user', 'content': message}]

    response = openai.chat.completions.create(model= MODEL, messages= messages, tools= tools)

    # print(response.choices[0].finish_reason)

    if response.choices[0].finish_reason == 'tool_calls':
        message = response.choices[0].message
        #print(message)
        response = handle_tool_call(message)
        #print(response)
        messages.append(message)
        messages.append(response)

        response = openai.chat.completions.create(model= MODEL, messages= messages)

    return response.choices[0].message.content

In [23]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):

    tool_call = message.tool_calls[0]
    #print(tool_call)

    if tool_call.function.name == 'get_ticket_price':
        arguments = json.loads(tool_call.function.arguments)
        #print(arguments)

        city = arguments.get('destination_city')
        #print(city)

        price_details = get_ticket_price(city)

        response = {
            'role': 'tool',
            'content': price_details,
            'tool_call_id': tool_call.id,
        }

    return response

In [29]:
gr.ChatInterface(fn = chat).launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


Tool called for city Paris


## Let's make a couple of improvements

### Above code implementation fails if:

1. We ask it for ticket prices to a couple of places instead of one with error:

`openai.BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages:`


To circumvent that, we can modify the code to make tool calls multiple times.

In [40]:
# No Changes Here:

def chat(message, history):

    history = [{'role': h['role'], 'content': h['content']} for h in history]

    messages = [{'role': 'system', 'content': system_message}] + history + [{'role': 'user', 'content': message}]

    response = openai.chat.completions.create(model= MODEL, messages= messages, tools= tools)

    if response.choices[0].finish_reason == 'tool_calls':

        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model= MODEL, messages= messages)

    return response.choices[0].message.content

In [41]:
# Change Here: Instead of checking just one tool call by message.tool_calls[0], we will run a for loop for all necessary tool calls.

def handle_tool_calls(message):

    responses = []

    for tool_call in message.tool_calls:
        if tool_call.function.name == 'get_ticket_price':
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                'role': 'tool',
                'content': price_details,
                'tool_call_id': tool_call.id
            })

    return responses

In [43]:
gr.ChatInterface(fn= chat).launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


Tool called for city berlin
Tool called for city tokyo
Tool called for city paris


### Changing the function to handle tool calls 1 after another:

In [46]:
def chat(message, history):

    history = [{'role': h['role'], 'content' :h['content']} for h in history]

    messages = [{'role': 'system', 'content': system_message}] + history + [{'role': 'user', 'content': message}]

    response = openai.chat.completions.create(model= MODEL, messages= messages, tools= tools)

    while response.choices[0].finish_reason== 'tool_calls':

        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

In [47]:
gr.ChatInterface(fn= chat).launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
Tool called for city Paris
